# 01_eda.ipynb
- rawデータの確認


## 準備

In [2]:
import os
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib_fontja  # noqa: F401

In [12]:
# rawデータのロード（国土交通省CSVはCP932/Shift_JISで配布されている）
df_raw = pd.read_csv('../data/raw/Tokyo_20251_20254.csv', encoding='cp932')
display(df_raw.head())

,種類,価格情報区分,市区町村コード,都道府県名,市区町村名,地区名,最寄駅：名称,最寄駅：距離（分）,取引価格（総額）,間取り,...,建築年,建物の構造,用途,今後の利用目的,都市計画,建ぺい率（％）,容積率（％）,取引時期,改装,取引の事情等
0,中古マンション等,成約価格情報,13101,東京都,千代田区,岩本町,小伝馬町,2,59000000,１ＬＤＫ,...,2014年,ＲＣ,NaN,NaN,商業,NaN,NaN,2025年第1四半期,NaN,NaN
1,中古マンション等,成約価格情報,13101,東京都,千代田区,岩本町,岩本町,4,120000000,２ＬＤＫ,...,2023年,ＲＣ,NaN,NaN,商業,NaN,NaN,2025年第1四半期,NaN,NaN
2,中古マンション等,成約価格情報,13101,東京都,千代田区,岩本町,岩本町,NaN,38000000,１ＤＫ,...,2007年,ＲＣ,NaN,NaN,商業,NaN,NaN,2025年第1四半期,NaN,NaN
3,中古マンション等,成約価格情報,13101,東京都,千代田区,岩本町,岩本町,3,65000000,２ＬＤＫ,...,2016年,ＲＣ,NaN,NaN,NaN,NaN,NaN,2025年第1四半期,NaN,NaN
4,中古マンション等,成約価格情報,13101,東京都,千代田区,九段北,市ケ谷,3,45000000,１Ｋ,...,2005年,鉄骨造,NaN,NaN,商業,NaN,NaN,2025年第1四半期,NaN,NaN


In [14]:
display(df_raw.isna().sum())

種類               0
価格情報区分           0
市区町村コード          0
都道府県名            0
市区町村名            0
地区名              0
最寄駅：名称          37
最寄駅：距離（分）      710
取引価格（総額）         0
間取り           1029
面積（㎡）            0
建築年            320
建物の構造          528
用途           28716
今後の利用目的      27207
都市計画           401
建ぺい率（％）      26763
容積率（％）       26763
取引時期             0
改装           27857
取引の事情等       39620
dtype: int64

In [16]:
# 欠損率と、距離列との重なりを見る
display(df_raw[['最寄駅：名称', '最寄駅：距離（分）', '市区町村名']].isna().sum())
display(df_raw[df_raw['最寄駅：名称'].isna()]['市区町村名'].value_counts().head(10))

最寄駅：名称        37
最寄駅：距離（分）    710
市区町村名          0
dtype: int64

市区町村名
港区      5
大田区     5
台東区     3
杉並区     3
新宿区     2
目黒区     2
世田谷区    2
練馬区     2
日野市     2
中央区     1
Name: count, dtype: int64

In [18]:
df_raw["用途"].value_counts()

用途
住宅        10822
事務所          58
店舗           40
その他          22
事務所、店舗        3
駐車場           2
住宅、その他        2
倉庫            1
Name: count, dtype: int64

In [19]:
df_raw["今後の利用目的"].value_counts()

今後の利用目的
住宅     11240
その他      982
事務所      186
店舗        51
Name: count, dtype: int64

In [20]:
# 両列がともに記入されている行で、対応関係を見る
both_filled = df_raw.dropna(subset=['用途', '今後の利用目的'])

# クロス集計
crosstab = pd.crosstab(
    both_filled['用途'],
    both_filled['今後の利用目的'],
    margins=True,
    normalize='index',  # 用途ごとの割合
)
print(crosstab)

# 用途='住宅' のとき、今後の利用目的の内訳
print(both_filled[both_filled['用途'] == '住宅']['今後の利用目的'].value_counts(normalize=True))

今後の利用目的       その他       事務所        住宅        店舗
用途                                             
その他      1.000000  0.000000  0.000000  0.000000
事務所      0.087719  0.771930  0.122807  0.017544
事務所、店舗   0.000000  0.000000  0.000000  1.000000
住宅       0.075106  0.009545  0.914674  0.000675
住宅、その他   0.000000  0.000000  1.000000  0.000000
倉庫       0.000000  1.000000  0.000000  0.000000
店舗       0.102564  0.000000  0.025641  0.871795
駐車場      1.000000  0.000000  0.000000  0.000000
All      0.077348  0.013717  0.904649  0.004287
今後の利用目的
住宅     0.914674
その他    0.075106
事務所    0.009545
店舗     0.000675
Name: proportion, dtype: float64


In [21]:
display(df_raw["都市計画"].value_counts())

都市計画
商業      11850
準工業      7199
１中住専     5207
近隣商業     4649
１種住居     4112
１低住専     1843
２種住居     1606
工業       1133
２中住専      910
準住居       620
２低住専      131
工業専用        5
Name: count, dtype: int64

In [22]:
display(df_raw["改装"].value_counts())

改装
未改装     7310
改装済み    4499
Name: count, dtype: int64